# UAV2UAV — YOLOX-s 4 類熱影像訓練 (Kaggle GPU)

**使用前置 (Notebook 右側設定)**:
1. Accelerator: **GPU T4 x2 或 P100** (擇一即可,本 notebook 用單卡)
2. Internet: **ON**
3. Add Input → 掛兩個 dataset:`uav2uav-code` (程式碼 zip)、`antiuav410` (Anti-UAV410.zip)

跑完後到 Output 下載 `work_dirs/yolox_s_uav2uav/latest.pth`,放回本機 `work_dirs/yolox_s_uav2uav/`,接著本機跑 `make export-onnx && make nef`。

版本鎖定與本機 docker 完全一致 (torch 1.12.1+cu113 / mmcv-full 1.6.0 / kneron-mmdetection)。

In [ ]:
%%bash
# Cell 1: 環境 — uv venv py3.10 + cu113 wheels (Kaggle 系統 python 太新,不能直接用)
set -e
pip -q install uv
export UV_PYTHON_INSTALL_DIR=/kaggle/working/uv-python
uv venv /kaggle/working/venv --python 3.10
PY=/kaggle/working/venv/bin
uv pip install -p $PY/python setuptools wheel
uv pip install -p $PY/python torch==1.12.1+cu113 torchvision==0.13.1+cu113 --index-url https://download.pytorch.org/whl/cu113
uv pip install -p $PY/python mmcv-full==1.6.0 -f https://download.openmmlab.com/mmcv/dist/cu113/torch1.12.0/index.html
git clone --depth 1 https://github.com/kneron/kneron-mmdetection /kaggle/working/kneron-mmdetection
cd /kaggle/working/kneron-mmdetection
uv pip install -p $PY/python -r requirements/build.txt
uv pip install -p $PY/python --no-build-isolation -e .
uv pip install -p $PY/python 'yapf==0.40.1'
$PY/python -c "import torch, mmcv, mmdet; print('torch', torch.__version__, torch.cuda.is_available(), '| mmcv', mmcv.__version__, '| mmdet', mmdet.__version__)"

In [ ]:
%%bash
# Cell 2: 程式碼 + 資料集 (Anti-UAV410 可選 — 沒掛載就只用 TU2U+HIT-UAV)
set -e
cd /kaggle/working
mkdir -p uav2uav && unzip -qo /kaggle/input/uav2uav-code/uav2uav-code.zip -d uav2uav
mkdir -p uav2uav/data/raw
[ -d uav2uav/data/raw/ThermalUAV2UAV_Dataset ] || git clone --depth 1 https://github.com/GabryV00/ThermalUAV2UAV_Dataset uav2uav/data/raw/ThermalUAV2UAV_Dataset
if [ ! -d uav2uav/data/raw/HIT-UAV ]; then
  wget -q -O /tmp/HIT-UAV.zip https://github.com/suojiashun/HIT-UAV-Infrared-Thermal-Dataset/releases/download/v1.2.1/HIT-UAV.zip
  unzip -q /tmp/HIT-UAV.zip -d uav2uav/data/raw/ && rm /tmp/HIT-UAV.zip
fi
if [ -d /kaggle/input/antiuav410 ] && [ ! -d uav2uav/data/raw/AntiUAV410/train ]; then
  mkdir -p uav2uav/data/raw/AntiUAV410
  unzip -q /kaggle/input/antiuav410/Anti-UAV410.zip -d uav2uav/data/raw/AntiUAV410/
else
  echo '(Anti-UAV410 未掛載,跳過 — 首跑用 TU2U+HIT-UAV;之後加掛重跑即併入)'
fi
du -sh uav2uav/data/raw/*

In [ ]:
%%bash
# Cell 3: 轉 COCO + merge
# Anti-UAV410 用 stride=30 (本機預設 10): 控制 epoch 時間 + 平衡空對空/地對空比例
set -e
PY=/kaggle/working/venv/bin/python
cd /kaggle/working/uav2uav/datasets
$PY convert_tu2u_to_coco.py --root ../data/raw/ThermalUAV2UAV_Dataset --out-dir ../data/coco/annotations
$PY convert_hituav_to_coco.py --root ../data/raw/HIT-UAV --out-dir ../data/coco/annotations
$PY convert_antiuav410_to_coco.py --root ../data/raw/AntiUAV410 --out-dir ../data/coco/annotations --stride 30
$PY merge_coco.py --ann-dir ../data/coco/annotations

In [ ]:
%%bash
# Cell 4: 訓練 (單 GPU)。Kaggle 單 session 上限 12h;40 epochs 在 T4 估 5~8h (估算)。
# 若被中斷: 重跑全部 cell,並把下行加上 --resume-from /kaggle/working/work_dirs/yolox_s_uav2uav/latest.pth
set -e
export KMM_ROOT=/kaggle/working/kneron-mmdetection
export UAV2UAV_ROOT=/kaggle/working/uav2uav
cd /kaggle/working/kneron-mmdetection
/kaggle/working/venv/bin/python tools/train.py \
  /kaggle/working/uav2uav/training/configs/yolox_s_uav2uav.py \
  --work-dir /kaggle/working/work_dirs/yolox_s_uav2uav \
  --cfg-options runner.max_epochs=40 data.workers_per_gpu=2 \
  custom_hooks.0.num_last_epochs=10 lr_config.num_last_epochs=10 evaluation.interval=5

In [ ]:
%%bash
# Cell 5: 收尾 — 只留 checkpoint 與 log 進 Output (Kaggle Output 上限 20GB,不要把資料集留在 working)
cd /kaggle/working
ls -lh work_dirs/yolox_s_uav2uav/ | tail -5
rm -rf uav2uav/data/raw kneron-mmdetection venv uv-python
echo '完成 — 到 Output 頁下載 work_dirs/yolox_s_uav2uav/latest.pth'